# Accessing Sample Digital Earth Normalised Radar Backscatter Products Using STAC Geoparquet

- Some sample and test products that are not published may be provided in a stac geoparquet file
- This is a lightweight solution of making cloud optimised geotiff (COG) products accessible
- Examples include Sentinel-1 Extra Wide (EW) NRB data made using the pyroSAR-GAMMA pipeline
- Once loaded into xarray, the same transformations presented in the other notebooks can be applied 

## Important STAC terminology
The STAC standard has four important components:

* **Catalog**: A structure for organising multiple datasets managed by a given provider.
* **Collection**: A structure for organising all items in a single dataset.
* **Item** A single spatio-temporal item, such as one observation in a dataset.
* **Asset** A single data measurement associated with an item, such as a single band.

For the **EW** Normalised Radar Backscatter data, it is important to note that a single *item* corresponds to a single *scene*.

## Set-up

### Import required libraries

In [ ]:
import stac_geoparquet
from odc.geo import BoundingBox
import geopandas as gpd
from odc.stac import load
import shapely
import fsspec
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import requests
import os

### Set the link the the STAC Geoparquet

In [ ]:
parquet_file_url = "https://data.dev.dea.ga.gov.au/experimental/baseline/mcmurdo/ga_s1_nrb_hh_hv_1.parquet"

## Load into a geopandas dataframe

In [ ]:
with fsspec.open(parquet_file_url, mode="rb") as f:
    gdf = gpd.read_parquet(f)
print(f'Number of items in gdf : {len(gdf)}')
print(f'Sample item in dataframe')
gdf.head(1)

## Plot coverage on map of antarctica

In [ ]:
# Get medium resolution coastline shapefile
url = "https://ramadda.data.bas.ac.uk/repository/entry/get/add_coastline_medium_res_line_v7_11.shp.zip?entryid=synth%3A333065a9-633d-4005-ae41-fb7ae5ae7a91%3AL2FkZF9jb2FzdGxpbmVfbWVkaXVtX3Jlc19saW5lX3Y3XzExLnNocC56aXA%3D"
antarctic_coast = gpd.read_file(url).to_crs(3031)
# Plot items by the collection
ax = gdf.to_crs(3031).plot(
    column="collection",
    categorical=True,
    figsize=(8, 8),
    legend=True
)
# Overlay coastline
antarctic_coast.plot(ax=ax, color="black", linewidth=1)
# Set legend title
leg = ax.get_legend()
leg.set_title("Collection")
plt.show()


## Filter items by collection, area of interest and time

In [ ]:
# Area of interest - Erebus Ice Tongue
collection = "ga_s1_nrb_ew_hh_hv_1"
start_time = "2019-01-01 12:05:53"
end_time = "2019-01-10 12:06:00"
aoi_bbox = BoundingBox(
    left=166.3,
    bottom=-77.75,
    right=167.4,
    top=-77.6, 
    crs="EPSG:4326"
)
aoi_shape = shapely.geometry.shape(aoi_bbox.polygon)


# filter by collection
filtered_items = gdf[gdf["collection"] == collection]
# filter by area
filtered_items = filtered_items[filtered_items.intersects(aoi_shape)]
# filter by time
filtered_items = gdf[
    (filtered_items.datetime >= start_time) & (filtered_items.datetime <= end_time)
]
print(f'Number of Items After Filtering : {len(filtered_items)}')

## Convert the geopandas dataframe to stac item collection

In [ ]:
stac_collection = stac_geoparquet.to_item_collection(filtered_items)
stac_collection[0]

## Load the items into an ODC dataset 

In [ ]:
ds = load(
    stac_collection,
    chunks={},
    resolution=20,
    crs="EPSG:3031",
    geopolygon=aoi_shape,
)
ds

### Plot the sample area

In [ ]:
ds.isel(time=0).hh_gamma0_db.plot.imshow(cmap="bone", vmin=-20, vmax=0, figsize=(8,5)) #uncomment to show sample plotted

## Optional - Downloading a full items / scenes

In [ ]:
# View the assets that can be downloaded
print(f'Available assets:')
print(list(stac_collection[0].assets.keys()))
# Settings for downloads
assets_to_download = ['hh_gamma0_db','hv_gamma0_db']
number_of_items_to_download = 1 # len(stac_collection)
download_folder = Path("data") # where to download files
make_subfolder_for_scene = True # add a subfolder for each scene / item_id
os.makedirs(download_folder, exist_ok=True)

In [ ]:
# iterate through the items and download
for i,item in enumerate(stac_collection):
    print(f"Item id ({i+1} of {number_of_items_to_download}): {item.id}")
    for asset in assets_to_download:
        asset_link = stac_collection[i].assets[asset].href
        asset_filename = Path(asset_link).name
        if make_subfolder_for_scene:
            out_path = download_folder / item.id / asset_filename
            os.makedirs(out_path.parent, exist_ok=True)
        else:
            out_path = download_folder / asset_filename
       
        print(f'downloading asset {asset} from link, maintaining file name : {asset_link}')
        with requests.get(asset_link, stream=True) as r:
            r.raise_for_status()
            with open(out_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        
        print(f'asset : {asset}, saved to {out_path}')

    if (i+1) == number_of_items_to_download:
        break